In [ ]:
!pip install -U transformers bitsandbytes accelerate sentence-transformers faiss-cpu huggingface_hub

In [ ]:
import faiss
import pickle
import torch
from transformers import pipeline, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer
from huggingface_hub import login

# 2. Authentication (Use your 'READ' token)
login(token="")

# 3. Setup Memory-Saving Brain (Gemma 3 4B)
model_id = "google/gemma-3-4b-it"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

print("Loading Gemma 3 4B onto Colab GPU...")
generator = pipeline(
    "text-generation",
    model=model_id,
    model_kwargs={
        "quantization_config": quantization_config,
        "device_map": "auto"
    }
)

# 4. Setup the Library Search (Using the files you just uploaded)
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
index = faiss.read_index("bank_faiss.index")
with open("text_mapping.pkl", "rb") as f:
    text_mapping = pickle.load(f)


Loading Gemma 3 4B onto Colab GPU...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# C. Generation (Robust Version for Colab)
def nust_bank_chat(user_query):
    # A. Retrieval (Requirement 3)
    query_vector = embed_model.encode([user_query])
    distances, indices = index.search(query_vector, k=3)
    context = "\n".join([text_mapping[i] for i in indices[0]])

    # B. Prompt Engineering (Requirement 2)
    # We use a very clear "Answer:" tag so we can split the text later
    # Updated Prompt for a cleaner, professional response
    prompt = f"""
    You are an official NUST Bank Assistant. Use the following context to answer the user's question in a full, professional sentence.
    Do not include technical tags like "Category" or "Topic" in your final answer.

    Context:
    {context}

    Question: {user_query}
    Answer:"""
    # C. Generation
    # We removed 'stop_sequence' to avoid the ValueError
    res = generator(
        prompt,
        max_new_tokens=100,
        temperature=0.1,
        do_sample=True,
        return_full_text=False, # This ensures we only get the NEW text
        pad_token_id=generator.tokenizer.eos_token_id
    )

    # D. Post-Processing
    # We manually stop the text if the model starts hallucinating new questions
    raw_answer = res[0]["generated_text"].strip()
    clean_answer = raw_answer.split("Question:")[0].split("\n\n")[0].strip()

    return clean_answer

# TEST CASE
print("--- TEST OUTPUT ---")
print(nust_bank_chat("What are the profit rates for a 1-year Term Deposit?"))

--- TEST OUTPUT ---


Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample', 'max_new_tokens', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Fr a 1-Year Term Depsit, the details are: Mnthly | 0.17 and Quarterly | 0.18.


In [ ]:
import os

# 1. Define save directories
save_directory_drive = "/content/drive/MyDrive/NUST_Bank_Gemma3"
save_directory_local = "/content/NUST_Bank_Gemma3_local"

# Create local directory if it doesn't exist
os.makedirs(save_directory_local, exist_ok=True)

# 2. Save the model first to Google Drive
generator.model.save_pretrained(save_directory_drive)

# 3. Save the model to local SSD
generator.model.save_pretrained(save_directory_local)

# 4. CLEAN the tokenizer's configuration (The Fix)
# We remove the BitsAndBytesConfig which is causing the JSON error
if hasattr(generator.tokenizer, "init_kwargs"):
    generator.tokenizer.init_kwargs.pop("quantization_config", None)

# 5. Now save the tokenizer to Google Drive
generator.tokenizer.save_pretrained(save_directory_drive)

# 6. Save the tokenizer to local SSD
generator.tokenizer.save_pretrained(save_directory_local)

print(f"Model and Tokenizer successfully saved to {save_directory_drive} (Google Drive) and {save_directory_local} (Local SSD)")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and Tokenizer successfully saved to /content/drive/MyDrive/NUST_Bank_Gemma3 (Google Drive) and /content/NUST_Bank_Gemma3_local (Local SSD)


Gradio app

In [ ]:
!pip install gradio -q

In [ ]:
# CELL 2
import faiss
import pickle
import torch
import warnings
from transformers import pipeline, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer
from google.colab import drive

warnings.filterwarnings("ignore")

print("1. Connecting to Google Drive...")
drive.mount('/content/drive')

print("2. Loading NUST Bank FAISS Database...")
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
index = faiss.read_index("bank_faiss.index")
with open("text_mapping.pkl", "rb") as f:
    text_mapping = pickle.load(f)

print("3. Loading Gemma 3 from Drive...")
drive_model_path = "/content/drive/MyDrive/NUST_Bank_Gemma3"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

generator = pipeline(
    "text-generation",
    model=drive_model_path,
    model_kwargs={
        "quantization_config": quantization_config,
        "device_map": "auto"
    }
)
print("Brain is online and locked in memory!")

1. Connecting to Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
2. Loading NUST Bank FAISS Database...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


3. Loading Gemma 3 from Drive...


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Brain is online and locked in memory!


In [ ]:
# CELL 3
import gradio as gr

def bank_assistant(message, history):
    # 1. Smarter Python Interception (Catches phrases, not just exact words)
    msg_lower = message.lower()
    greeting_words = ["hi", "hello", "hey", "salam", "assalam"]

    if any(word in msg_lower for word in greeting_words) or "thank" in msg_lower:
        return "Hello! Welcome to NUST Bank. How can I assist you with our services today?"

    # 2. Search Database
    query_vector = embed_model.encode([message])
    distances, indices = index.search(query_vector, k=3)
    context = "\n".join([text_mapping[i] for i in indices[0]])

    # DEBUG: Prints the retrieved data to your Colab terminal so you can verify it
    print(f"\n--- RETRIEVED CONTEXT FOR: '{message}' ---")
    print(context)
    print("-------------------------------------------\n")

    # 3. Better Prompt Formatting
    messages = [
        {
            "role": "system",
            "content": (
                "You are a professional NUST Bank AI Assistant. Your job is to answer questions using ONLY the provided Context.\n"
                "- If the answer is in the Context, summarize it clearly and bold important numbers or profit rates.\n"
                "- If the answer is completely missing from the Context, say exactly: 'I apologize, but I do not have that specific information in my records.'"
            )
        },
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {message}"
        }
    ]

    # 4. Generate Answer
    prompt = generator.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    res = generator(
        prompt,
        max_new_tokens=150,
        temperature=0.1,
        do_sample=True,
        return_full_text=False
    )

    return res[0]["generated_text"].strip()


In [ ]:
# --- Automated QA Testing for NUST Bank AI ---

print("🚀 Starting Automated QA Tests for NUST Bank Assistant...\n")

# 1. Define our test cases based on the 3 categories
test_queries = {
    "🟢 CORE RAG (Data Retrieval)": [
        "What are the profit rates for a 1-year Term Deposit?",
        "What is the minimum balance required to open a standard savings account?"
    ],
    "🟡 SMALL TALK (Rule 1: Greeting Override)": [
        "Hi there!",
        "Thank you, that was very helpful."
    ],
    "🔴 HALLUCINATION SHIELD (Rule 2: Out of Domain)": [
        "Who is currently leading the Formula 1 driver standings?",
        "What is the recipe for a chocolate cake?"
    ]
}

# 2. Loop through the tests and grade the AI
for category, questions in test_queries.items():
    print(f"{"="*50}")
    print(f"🧪 TESTING CATEGORY: {category}")
    print(f"{"="*50}\n")

    for q in questions:
        print(f"👤 USER: {q}")

        # We pass an empty list [] for history since this is a fresh test
        try:
            ai_response = bank_assistant(q, [])
            print(f"🤖 BANK AI: {ai_response}\n")
        except Exception as e:
            print(f"❌ ERROR: {e}\n")

print("✅ Automated Testing Complete! Review the outputs above.")

Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🚀 Starting Automated QA Tests for NUST Bank Assistant...

🧪 TESTING CATEGORY: 🟢 CORE RAG (Data Retrieval)

👤 USER: What are the profit rates for a 1-year Term Deposit?

--- RETRIEVED CONTEXT FOR: 'What are the profit rates for a 1-year Term Deposit?' ---
Category: Rate Sheet | Topic: Profit Rates | Question: What is the prfit rate and payut fr PLS Savings? | Answer: Fr PLS Savings, the details are: One Year | 0.175
Category: Rate Sheet | Topic: Profit Rates | Question: What is the prfit rate and payut fr NUST Sahar - Term Depsit? | Answer: Fr NUST Sahar - Term Depsit, the details are: One Year | Mnthly | 0.17
Category: Rate Sheet | Topic: Profit Rates | Question: What is the prfit rate and payut fr PLS Savings? | Answer: Fr PLS Savings, the details are: Mnthly | 0.19 | Tw Years | Mnthly/ Quarterly/ Annual | 0.13
-------------------------------------------



Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 BANK AI: For NUST Sahar - Term Depsit, the details are: One Year | Mnthly | 0.17

👤 USER: What is the minimum balance required to open a standard savings account?

--- RETRIEVED CONTEXT FOR: 'What is the minimum balance required to open a standard savings account?' ---
Category: VP-BA | Topic: VP-BA | Question: Please tell me abut minimum Mnthly balance required ? | Answer: N initial depsit requirement Rs. 25,000/- (Mnthly Average Balance)
Category: VPCA | Topic: VPCA | Question: What is the pening and minimum balance requirement fr VPCA? | Answer: - N Opening Balance Requirements - N Minimum Mnthly Average Balance Requirements - Allcatin f Lckers (upn availability) Subsequent Charges/Annual Fees wuld be as per prevailing SOC On maintaining a minimum mnthly average balance f Rs. 50,000
Category: VPBA | Topic: VPBA | Question: Please tell me abut the Minimum Mnthly balance required fr this accunt? | Answer: N initial depsit requirement Rs. 100,000/- (Mnthly Average Balance)
----------

Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 BANK AI: I apologize, but I do not have that specific information in my records.

🧪 TESTING CATEGORY: 🟡 SMALL TALK (Rule 1: Greeting Override)

👤 USER: Hi there!
🤖 BANK AI: Hello! Welcome to NUST Bank. How can I assist you with our services today?

👤 USER: Thank you, that was very helpful.
🤖 BANK AI: Hello! Welcome to NUST Bank. How can I assist you with our services today?

🧪 TESTING CATEGORY: 🔴 HALLUCINATION SHIELD (Rule 2: Out of Domain)

👤 USER: Who is currently leading the Formula 1 driver standings?

--- RETRIEVED CONTEXT FOR: 'Who is currently leading the Formula 1 driver standings?' ---
Category: NUST4Car | Topic: NUST4Car | Question: 3. Which types f Vehicles are ffered under NUST aut finance? | Answer: Lcally assembled/manufactured New/Used vehicles (Cmmercial vehicles r vehicles fr cmmercial use cannt be financed)
Category: LCA | Topic: LCA | Question: What is the accunt type f Little Champs Accunt is it saving r current ? | Answer: This accunt is ffered bth in current and

Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🤖 BANK AI: I apologize, but I do not have that specific information in my records.

👤 USER: What is the recipe for a chocolate cake?

--- RETRIEVED CONTEXT FOR: 'What is the recipe for a chocolate cake?' ---
Category: PMYB &ALS | Topic: PMYB &ALS | Question: What are the Lan Limits f PMYB & ALS? | Answer: Tier 2 Abve Rs. 0.5M and up t Rs 1.5M Tier 3 Abve Rs 1.5 M and up t Rs 7.5M
Category: PMYB &ALS | Topic: PMYB &ALS | Question: Des the bank require any Equity fr this scheme? | Answer: Tier 2 10% (New Business) Tier 3 20% (New Business) Existing Business Nil
Category: PMYB &ALS | Topic: PMYB &ALS | Question: Is there any quta fr wmens? | Answer: 25% f the lans will g t wmen brrwers
-------------------------------------------

🤖 BANK AI: I apologize, but I do not have that specific information in my records.

✅ Automated Testing Complete! Review the outputs above.


In [ ]:
# Launch the User Interface
print("Launching User Interface...")
demo = gr.ChatInterface(
    fn=bank_assistant,
    title="🏦 NUST Bank AI Assistant",
    description="Ask me about NUST Bank's profit rates, term deposits, and policies!",
    theme="ocean",
)

demo.launch(share=True)

Launching User Interface...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a2a170443af45761e9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
